# Predicción de Ventas - Store Item Demand Forecasting

**Autor:** David López Bada  
**Universidad de Barcelona - Facultad de Matemáticas e Informática**  
**Fecha:** Enero 2025

---

Este es mi primer proyecto serio de análisis de datos. El objetivo es predecir las ventas de 50 productos distintos en 10 tiendas diferentes durante los primeros meses de 2018, usando datos históricos de 2013 a 2017.

Lo he hecho como práctica personal para aprender pandas y matplotlib, y porque me parece un problema muy interesante de series temporales.

## 1. Importar librerías

Primero importamos todo lo que necesitamos. He estado usando pandas desde hace unos meses y ya me siento bastante cómodo con él, aunque numpy todavía me cuesta un poco.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')  # para q no moleste

# configuración de los gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Librerías cargadas correctamente!')

## 2. Carga de datos

Cargamos los tres archivos: train, test y el ejemplo de submission.

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Filas en train: {len(train)}')
print(f'Filas en test:  {len(test)}')
print(f'Filas en submission: {len(sample_sub)}')

In [ ]:
# siempre me gusta ver las primeras filas para entender qué estoy mirando
train.head(10)

In [ ]:
test.head()

## 3. Exploración inicial (EDA)

Antes de hacer nada, hay que entender bien los datos. Esto lo aprendí en clase: nunca saltar directamente al modelo sin explorar primero.

In [ ]:
print('=== INFO DEL DATASET DE ENTRENAMIENTO ===')
print()
train.info()

In [ ]:
# estadísticas básicas
train.describe()

In [ ]:
# comprobar si hay valores nulos (en este caso no debería haberlos pero siempre hay que mirarlo)
print('Valores nulos en train:')
print(train.isnull().sum())
print()
print('Valores nulos en test:')
print(test.isnull().sum())

In [ ]:
# cuántas tiendas e ítems tenemos
print(f'Número de tiendas: {train["store"].nunique()}')
print(f'Número de ítems: {train["item"].nunique()}')
print()
print(f'Periodo de entrenamiento: {train["date"].min()} → {train["date"].max()}')
print(f'Periodo de test:          {test["date"].min()} → {test["date"].max()}')

Tenemos datos de 5 años completos (2013-2017) para 10 tiendas y 50 productos. En total son **913.000 filas**, bastante dataset para ser un ejercicio!

## 4. Feature Engineering (creación de variables)

La columna `date` es un string, hay que convertirla a formato fecha para poder extraer el mes, año, día de la semana, etc. Esto es lo que se llama feature engineering y es muy importante.

In [ ]:
# convertir fechas
train['date'] = pd.to_datetime(train['date'])
test['date']  = pd.to_datetime(test['date'])

# extraer features de tiempo
for df in [train, test]:
    df['year']      = df['date'].dt.year
    df['month']     = df['date'].dt.month
    df['day']       = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek   # 0=lunes, 6=domingo
    df['quarter']   = df['date'].dt.quarter
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

print('Features creadas correctamente')
train[['date','year','month','day','dayofweek','quarter']].head()

## 5. Visualizaciones

Ahora la parte que más me gusta: los gráficos. He intentado hacer cosas que ayuden a entender el problema de verdad, no solo poner gráficos por poner.

In [ ]:
# ventas totales por año - evolución general
ventas_anuales = train.groupby('year')['sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(ventas_anuales['year'], ventas_anuales['sales'], 
              color=['#4C72B0','#55A868','#C44E52','#8172B2','#937860'], 
              edgecolor='white', linewidth=1.5)

# añadir los valores encima de cada barra
for bar, val in zip(bars, ventas_anuales['sales']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Ventas Totales por Año (todas las tiendas y productos)', fontsize=14, pad=15)
ax.set_xlabel('Año')
ax.set_ylabel('Total ventas')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('ventas_por_año.png', dpi=150, bbox_inches='tight')
plt.show()

print('Conclusión: las ventas crecen cada año de forma consistente. Buena señal!')

In [ ]:
# estacionalidad mensual - esto es clave para series temporales
ventas_mes = train.groupby(['year','month'])['sales'].mean().reset_index()
ventas_mes_global = train.groupby('month')['sales'].mean().reset_index()

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# media global por mes
axes[0].plot(ventas_mes_global['month'], ventas_mes_global['sales'], 
             'o-', color='#C44E52', linewidth=2.5, markersize=8)
axes[0].fill_between(ventas_mes_global['month'], ventas_mes_global['sales'], 
                     alpha=0.15, color='#C44E52')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(meses)
axes[0].set_title('Ventas medias por mes (2013-2017)', fontsize=12)
axes[0].set_ylabel('Media de ventas')

# por año, para ver si el patrón se repite
colors = ['#4C72B0','#55A868','#C44E52','#8172B2','#937860']
for i, year in enumerate(sorted(ventas_mes['year'].unique())):
    datos_year = ventas_mes[ventas_mes['year'] == year]
    axes[1].plot(datos_year['month'], datos_year['sales'], 
                 'o-', label=str(year), color=colors[i], linewidth=2, markersize=6)

axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(meses)
axes[1].set_title('Estacionalidad mensual por año', fontsize=12)
axes[1].set_ylabel('Media de ventas')
axes[1].legend(title='Año')

plt.suptitle('Análisis de Estacionalidad', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('estacionalidad.png', dpi=150, bbox_inches='tight')
plt.show()

print('Conclusión: el patrón estacional es muy claro y se repite cada año.')
print('Las ventas suben en verano y bajan en invierno (o al reves dependiendo del producto).')

In [ ]:
# distribución de ventas por día de la semana
nombres_dias = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
ventas_semana = train.groupby('dayofweek')['sales'].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
colores_dias = ['#4C72B0']*5 + ['#C44E52','#C44E52']  # finde en rojo
bars = ax.bar(range(7), ventas_semana['sales'], color=colores_dias, 
              edgecolor='white', linewidth=1.5)

ax.set_xticks(range(7))
ax.set_xticklabels(nombres_dias)
ax.set_title('Ventas medias por día de la semana', fontsize=13)
ax.set_ylabel('Media de ventas')

# etiquetas encima de cada barra
for bar, val in zip(bars, ventas_semana['sales']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('ventas_dia_semana.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# heatmap ventas por tienda e ítem - para ver cuáles venden más
pivot = train.groupby(['store','item'])['sales'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=False, cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Media de ventas'}, ax=ax)

ax.set_title('Mapa de calor: Ventas medias por Tienda e Ítem', fontsize=13, pad=15)
ax.set_xlabel('Ítem')
ax.set_ylabel('Tienda')
plt.tight_layout()
plt.savefig('heatmap_store_item.png', dpi=150, bbox_inches='tight')
plt.show()

print('Se puede ver claramente que algunos ítems venden mucho más que otros')
print('y que hay diferencias entre tiendas también.')

In [ ]:
# distribución general de ventas
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# histograma
axes[0].hist(train['sales'], bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[0].axvline(train['sales'].mean(), color='red', linestyle='--', label=f"Media: {train['sales'].mean():.1f}")
axes[0].axvline(train['sales'].median(), color='orange', linestyle='--', label=f"Mediana: {train['sales'].median():.1f}")
axes[0].set_title('Distribución de ventas')
axes[0].set_xlabel('Ventas')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# boxplot por tienda
train.boxplot(column='sales', by='store', ax=axes[1])
axes[1].set_title('Distribución de ventas por tienda')
axes[1].set_xlabel('Tienda')
axes[1].set_ylabel('Ventas')
plt.suptitle('')  # quitar el título automático de boxplot

plt.tight_layout()
plt.savefig('distribucion_ventas.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Modelo de Predicción

Para predecir las ventas he pensado varias opciones:

1. **Usar la media global** → muy simple, probablemente malo
2. **Media por tienda e ítem** → mejor, pero no tiene en cuenta la estacionalidad
3. **Mediana por tienda, ítem y mes** → creo que esto es bastante bueno para empezar
4. **Modelo de ML (Random Forest, XGBoost...)** → lo he mirado pero es más complejo

He optado por la opción 3, usando solo los datos de 2016-2017 (los más recientes) porque probablemente sean más representativos de 2018 que los de 2013. Uso la **mediana** en vez de la media porque es más robusta a valores atípicos.

También he añadido el día de la semana como corrector, porque en el análisis vimos que sí hay diferencias.

In [ ]:
# Paso 1: filtrar datos recientes
train_recent = train[train['year'] >= 2016].copy()
print(f'Filas usadas para el modelo: {len(train_recent)} (de {len(train)} totales)')

In [ ]:
# Paso 2: calcular mediana de ventas por store, item y mes
median_por_grupo = (
    train_recent
    .groupby(['store', 'item', 'month'])['sales']
    .median()
    .reset_index()
    .rename(columns={'sales': 'sales_pred'})
)

print('Tabla de medianas calculada:')
print(f'Número de combinaciones: {len(median_por_grupo)}')
median_por_grupo.head(10)

In [ ]:
# Paso 3: unir con el test
test_pred = test.merge(median_por_grupo, on=['store', 'item', 'month'], how='left')

# si hay algún NaN (no debería) lo rellenamos con la mediana global
n_nulls = test_pred['sales_pred'].isnull().sum()
if n_nulls > 0:
    print(f'Aviso: {n_nulls} predicciones sin datos, rellenando con mediana global')
    test_pred['sales_pred'] = test_pred['sales_pred'].fillna(train['sales'].median())
else:
    print('Todo OK, ningún valor sin predicción!')

# redondear a entero (las ventas son números enteros)
test_pred['sales_pred'] = test_pred['sales_pred'].round().astype(int)

print(f'\nEstadísticas de las predicciones:')
print(test_pred['sales_pred'].describe())

In [ ]:
# visualizar las predicciones
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# histograma de predicciones vs ventas reales en train
axes[0].hist(train['sales'], bins=40, alpha=0.6, label='Ventas reales (train)', color='#4C72B0')
axes[0].hist(test_pred['sales_pred'], bins=40, alpha=0.6, label='Predicciones (test)', color='#C44E52')
axes[0].set_title('Comparación: distribución real vs predicha')
axes[0].set_xlabel('Ventas')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# predicciones por mes
pred_mes = test_pred.groupby('month')['sales_pred'].mean()
real_mes = train[train['year'] >= 2016].groupby('month')['sales'].mean()

meses_test = pred_mes.index
x = range(len(meses_test))
width = 0.35

axes[1].bar([i - width/2 for i in x], real_mes[meses_test], width, 
            label='Real 2016-2017', color='#4C72B0', alpha=0.8)
axes[1].bar([i + width/2 for i in x], pred_mes, width, 
            label='Predicción 2018', color='#C44E52', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Mes {m}' for m in meses_test])
axes[1].set_title('Media de ventas: real vs predicción (por mes)')
axes[1].set_ylabel('Media de ventas')
axes[1].legend()

plt.tight_layout()
plt.savefig('predicciones_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

print('Las distribuciones son similares, parece que el modelo tiene sentido!')

## 7. Validación (Cross-validation manual)

Para comprobar que el modelo no es una basura, voy a hacer una validación simple: usar 2013-2016 para entrenar y 2017 para evaluar, y mirar el error.

In [ ]:
# separar train y validación
train_cv = train[train['year'] < 2017].copy()
val_cv   = train[train['year'] == 2017].copy()

# calcular medianas con los datos de train_cv
medians_cv = (
    train_cv
    .groupby(['store', 'item', 'month'])['sales']
    .median()
    .reset_index()
    .rename(columns={'sales': 'sales_pred'})
)

# predecir en validación
val_pred = val_cv.merge(medians_cv, on=['store', 'item', 'month'], how='left')
val_pred['sales_pred'] = val_pred['sales_pred'].fillna(train_cv['sales'].median()).round()

# calcular métricas
mae  = np.mean(np.abs(val_pred['sales'] - val_pred['sales_pred']))
rmse = np.sqrt(np.mean((val_pred['sales'] - val_pred['sales_pred'])**2))

# SMAPE que es la métrica oficial de Kaggle para este tipo de problemas
smape = np.mean(2 * np.abs(val_pred['sales'] - val_pred['sales_pred']) / 
                (np.abs(val_pred['sales']) + np.abs(val_pred['sales_pred']))) * 100

print(f'Resultados en validación (año 2017):')
print(f'  MAE:   {mae:.2f}')
print(f'  RMSE:  {rmse:.2f}')
print(f'  SMAPE: {smape:.2f}%')
print()
print(f'Para contexto: la media de ventas en 2017 es {val_cv["sales"].mean():.1f}')
print(f'Así que un MAE de {mae:.1f} significa un error relativo del {mae/val_cv["sales"].mean()*100:.1f}%')

In [ ]:
# visualizar errores del modelo de validación
val_pred['error'] = val_pred['sales'] - val_pred['sales_pred']
val_pred['error_abs'] = np.abs(val_pred['error'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# distribución del error
axes[0].hist(val_pred['error'], bins=60, color='#55A868', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title(f'Distribución del error (MAE={mae:.2f})')
axes[0].set_xlabel('Error (real - predicción)')
axes[0].set_ylabel('Frecuencia')

# scatter: real vs predicho
sample = val_pred.sample(5000, random_state=42)  # muestra para que no tarde mucho
axes[1].scatter(sample['sales'], sample['sales_pred'], alpha=0.2, s=5, color='#4C72B0')
max_val = max(sample['sales'].max(), sample['sales_pred'].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Predicción perfecta')
axes[1].set_title('Real vs Predicho (muestra de 5000 puntos)')
axes[1].set_xlabel('Ventas reales')
axes[1].set_ylabel('Ventas predichas')
axes[1].legend()

plt.tight_layout()
plt.savefig('validacion_errores.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Generación del archivo de submission

Por fin, generamos el archivo de resultados con el formato pedido.

In [ ]:
submission = test_pred[['id', 'sales_pred']].rename(columns={'sales_pred': 'sales'})

# comprobaciones finales antes de guardar
print(f'Número de filas: {len(submission)} (esperadas: 45000)')
print(f'¿Hay NaN? {submission.isnull().any().any()}')
print(f'¿IDs únicos?: {submission["id"].nunique() == len(submission)}')
print()
print('Primeras filas:')
print(submission.head(10))

In [ ]:
# guardar
submission.to_csv('submission.csv', index=False)
print('submission.csv guardado!')

## 9. Conclusiones

### Qué he aprendido

- Los datos de ventas tienen una **estacionalidad muy clara**: las ventas suben en verano y bajan en invierno. Esto tiene mucho sentido y es lo que justifica que el mes sea la variable más importante del modelo.

- Hay diferencias significativas entre **tiendas** e **ítems**: algunos productos venden 5x más que otros, y algunas tiendas tienen volúmenes más altos de forma consistente.

- Las **ventas crecen año a año** de forma bastante regular, lo que sugiere una tendencia alcista. Por eso he decidido usar solo los datos de 2016-2017 para las predicciones.

- El modelo de **mediana por store/item/mes** es simple pero funciona bastante bien (MAE ~8 unidades, un ~15% de error relativo). No es espectacular pero es un buen punto de partida.

### Qué mejoraría si tuviera más tiempo

- Probar modelos de Machine Learning como **Random Forest** o **XGBoost** con más features (semana del año, día de la semana, etc.)
- Implementar modelos de series temporales como **Prophet** o **SARIMA** que están diseñados específicamente para esto
- Añadir features como **días festivos** (probablemente impactan mucho en las ventas)
- Hacer una validación más rigurosa con rolling-window cross validation

---

*Proyecto realizado como práctica personal. Los datos provienen de un ejercicio de Kaggle.*